# Diachronic analysis — vocabulary & framing shift in parental-alienation case law

**Separate from the RAG pipeline (`src/rag_echr_ris.ipynb`).** This notebook answers a question retrieval-augmented generation structurally cannot.

> **"How did X change over time" is a distributional question over the whole corpus stratified by year.** Top-k retrieval **discards the distribution by design** — it returns the few most similar passages, not a representative sample — so the RAG pipeline **abstains** on such questions and **delegates** them here.

This notebook **surfaces candidate signals of change** — frequency, collocation, keyness — across two time bins (default 2015–2019 vs 2020–2025). It **does not prove that legal interpretation changed**; that claim is made only by **reading the cases the signals flag**. Every output below is a shortlist for doctrinal reading, not a finding.

**Scope (this run):** `LANG_SCOPE="EN"` → ECHR English judgments/decisions. The German RIS study is wired behind the same switch but **deferred**: its *Rechtssätze* (abstract legal principles) carry no single decision date — verified, the date field is the latest *applying* decision while the id encodes a 1953–2018 *originating* decision — so they cannot be validly time-binned. The two languages are **never pooled**; they are two separate studies run by flipping the switch.

## 1. Configuration

Paths are relative to the project root. `CORPUS_PATH` is a **frozen snapshot** — the notebook runs against a fixed file (point it at `archive/` for strict immutability) and prints a content fingerprint so a result set is traceable to a corpus state. `LANG_SCOPE` is the only switch that changes the study; the two languages are never combined.

In [ ]:
# --- imports (core path uses only stdlib + numpy/pandas/matplotlib) ---
from pathlib import Path
import re, json, math, hashlib
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- project paths (root = parent of src/) ---
ROOT = Path.cwd()
if ROOT.name == "src":
    ROOT = ROOT.parent
DATA, FIGURES, REPORTS = ROOT / "data", ROOT / "figures", ROOT / "reports"

# === LANG_SCOPE: two SEPARATE studies, NEVER pooled. Flip to run the other. ===
LANG_SCOPE = "EN"      # "EN" -> ECHR/English (active);  "DE" -> RIS/Austrian German (wired, deferred)

# === temporal binning ===
DATE_RANGE   = (2000, 2025)    # inclusive (corpus extended to 2000 on 2026-07-07)
BIN_BOUNDARY = 2013            # early = [2000, 2012] ; late = [2013, 2025]
EARLY_LABEL  = "%d-%d" % (DATE_RANGE[0], BIN_BOUNDARY - 1)
LATE_LABEL   = "%d-%d" % (BIN_BOUNDARY, DATE_RANGE[1])

# === genre-confound toggles ===
DROP_COMMUNICATED = True   # EN/ECHR: drop Registry-authored communicated-case templates
DROP_CRIMINAL     = True   # DE/RIS : drop criminal 'Entfremdung' (misappropriation) homonym

# === per-language corpus config ===
SOURCES = {
    "EN": {"name": "ECHR (English judgments & decisions)",
           "path": DATA / "echr_parental_alienation.json"},
    "DE": {"name": "RIS / OGH (Austrian German, Text decisions only)",
           "path": DATA / "ris_parental_alienation.json"},
}
CFG = SOURCES[LANG_SCOPE]
CORPUS_PATH = Path(CFG["path"])     # frozen snapshot

# === concept clusters (a concept is NOT one keyword) ===
CLUSTERS = {
    "EN": {"parental_alienation": [
        "alienat*", "estrang*", "parental alienation", "loyalty conflict",
        "turning the child against", "manipulat*", "undermin*"]},
    "DE": {"parental_alienation": [
        "entfremd*", "loyalitätskonflikt", "bindungstoleranz",
        "bindungsintoleranz", "wohlverhaltensgebot", "vereinnahm*"]},
}

# === stopwords (inline; no nltk download) ===
_EN_STOP = ("the a an and or of to in on at for with without by from as is are was were be been being "
            "this that these those it its their his her them they he she we you i me my our your "
            "not no nor but if then than so such which who whom whose what when where why how "
            "all any both each few more most other some only own same too very can will just "
            "should now also has have had do does did would could may might must shall "
            "against between into through during before after above below up down out off over under "
            "again further here there once because about upon per s t u re ll ve "
            "court applicant applicants article convention paragraph paragraphs case cases government "
            "mr mrs ms see art para pp vol")
_DE_STOP = ("der die das und oder von zu in den dem des ein eine einer einem eines ist sind war waren "
            "werden wird auf mit für ohne durch als auch nicht kein keine nur bei im am vom zum zur "
            "dass weil wenn dann so dieser diese dieses jener man sich nach vor über unter aus um gegen "
            "bis seit während wegen sowie bzw es er sie wir ihr du ich mich mir dir ihm ihn ihnen "
            "hat haben hatte wurde worden sein seine seiner seinem ihre ihren ihrer dessen deren "
            "ris dokument gericht ogh entscheidungsdatum geschäftszahl norm rechtssatz kopf "
            "european law identifier ecli rs te")
STOP = set((_EN_STOP if LANG_SCOPE == "EN" else _DE_STOP).split())

print("LANG_SCOPE =", LANG_SCOPE, "->", CFG["name"])
print("CORPUS_PATH =", CORPUS_PATH)
print("bins:  early", EARLY_LABEL, " | late", LATE_LABEL, " (boundary", BIN_BOUNDARY, ")")
print("stopwords:", len(STOP))

LANG_SCOPE = EN -> ECHR (English judgments & decisions)
CORPUS_PATH = /Users/maksimsmirnov/Desktop/thesis/data/echr_parental_alienation.json
bins:  early 2000-2012  | late 2013-2025  (boundary 2013 )
stopwords: 141


## 2. Concept clusters & tokeniser

A concept cluster maps **many surface forms** to one idea. Anchors support `*` stem wildcards and multiword phrases, matched with word-boundary regex (Unicode-aware, so German umlauts work). Phrases are tried before single-word stems so `parental alienation` is counted once, not also as `alienat*`.

In [ ]:
TOKEN_RE = re.compile(r"\w+", re.UNICODE)
def tokens(text):
    return TOKEN_RE.findall((text or "").lower())

def _anchor_pattern(anchor):
    a = anchor.strip().lower()
    if " " in a:                                   # multiword phrase
        return r"\b" + r"\s+".join(re.escape(p) for p in a.split()) + r"\b"
    if a.endswith("*"):                            # stem wildcard
        return r"\b" + re.escape(a[:-1]) + r"\w*"
    return r"\b" + re.escape(a) + r"\b"            # exact word

def _token_pattern(anchor):
    a = anchor.strip().lower()
    if a.endswith("*"):
        return re.compile(r"^" + re.escape(a[:-1]) + r"\w*$", re.IGNORECASE)
    return re.compile(r"^" + re.escape(a) + r"$", re.IGNORECASE)

CLUSTER = CLUSTERS[LANG_SCOPE]["parental_alienation"]
_pats = sorted((_anchor_pattern(a) for a in CLUSTER), key=len, reverse=True)   # phrases first
COMBINED = re.compile("|".join("(?:%s)" % p for p in _pats), re.IGNORECASE)
SINGLE_ANCHORS = [a for a in CLUSTER if " " not in a]
_SINGLE_RE = [_token_pattern(a) for a in SINGLE_ANCHORS]

def cluster_count(text):
    return sum(1 for _ in COMBINED.finditer(text or ""))      # non-overlapping occurrences
def is_anchor_token(tok):
    return any(r.match(tok) for r in _SINGLE_RE)

# self-test
_demo = ("The mother kept alienating the child and undermined contact; this parental alienation "
         "and loyalty conflict estranged the father."
         if LANG_SCOPE == "EN" else
         "Die Entfremdung des Kindes und der Loyalitätskonflikt verletzen das Wohlverhaltensgebot.")
print("cluster anchors:", CLUSTER)
print("self-test occurrences:", cluster_count(_demo))
print("anchor tokens in demo:", [t for t in tokens(_demo) if is_anchor_token(t)])

cluster anchors: ['alienat*', 'estrang*', 'parental alienation', 'loyalty conflict', 'turning the child against', 'manipulat*', 'undermin*']
self-test occurrences: 5
anchor tokens in demo: ['alienating', 'undermined', 'alienation', 'estranged']


## 3. Load → date → genre filter → bin

Date validity was checked before building (see the conversation): ECHR dates come from `judgementdate → ecli → referencedate`; RIS Text dates come from the decision id `JJT_YYYYMMDD` (= `entscheidungsdatum`, verified identical). The loader prints docs per year and per bin and **warns if a bin is thinner than 20 docs**. If the data file is absent it prints instructions instead of crashing.

In [ ]:
_DMY   = re.compile(r"(\d{1,2})/(\d{1,2})/(\d{4})")
_ECLIY = re.compile(r"ECLI:CE:ECHR:(\d{4}):")
_FTY   = re.compile(r"(?:communicated on|judgment\s+strasbourg|decision\s+strasbourg|strasbourg,?)\s+\d{1,2}\s+[A-Za-z]+\s+(\d{4})", re.IGNORECASE)
_RISID = re.compile(r"JJ[RT]_(\d{4})(\d{2})(\d{2})")
_OS    = re.compile(r"\d+\s*Os\b")
_STGB  = re.compile(r"\bStGB\b")

def echr_year(rec):
    m = _DMY.match((rec.get("judgementdate") or "").strip())
    if m: return int(m.group(3))
    m = _ECLIY.search(rec.get("ecli") or "")
    if m: return int(m.group(1))
    m = _DMY.match((rec.get("referencedate") or "").strip())
    if m: return int(m.group(3))
    m = _FTY.search((rec.get("full_text") or "")[:400])     # only needed if communicated kept
    if m: return int(m.group(1))
    return None

def echr_is_communicated(rec):
    if (rec.get("doctypebranch") or "") == "COMMUNICATEDCASES": return True
    if (rec.get("conclusion") or "").strip().lower() == "communicated": return True
    return (rec.get("full_text") or "")[:40].lower().startswith("communicated on")

def ris_year(rec):
    m = _RISID.match(rec.get("id") or "")
    return int(m.group(1)) if m else None

def ris_is_criminal(rec):
    if "strafrecht" in (rec.get("rechtsgebiete") or "").lower(): return True
    if _STGB.search(rec.get("normen") or ""): return True
    if _OS.search(rec.get("geschaeftszahl") or ""): return True
    return False

def _read_json(p):
    with open(p) as f:
        return json.load(f)

def corpus_fingerprint(p):
    h = hashlib.sha1()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()[:12]

def normalise(rec):
    if LANG_SCOPE == "EN":
        return {"id": rec.get("itemid") or rec.get("stable_id"),
                "year": echr_year(rec), "text": rec.get("full_text") or "",
                "genre_confound": echr_is_communicated(rec), "keep_type": True}
    return {"id": rec.get("id") or rec.get("stable_id"),
            "year": ris_year(rec), "text": rec.get("full_text") or "",
            "genre_confound": ris_is_criminal(rec),
            "keep_type": (rec.get("dokumenttyp") or "") == "Text"}

def load_corpus():
    print("LANG_SCOPE =", LANG_SCOPE, "->", CFG["name"])
    if not CORPUS_PATH.exists():
        print()
        print("[!] data file not found:", CORPUS_PATH)
        print("    This notebook reads a frozen snapshot written by src/master_import.ipynb.")
        print("    Run that import first, or set CORPUS_PATH to an archived copy, then re-run.")
        return pd.DataFrame(columns=["id", "year", "text", "bin", "n_tokens", "hits", "mentions"])
    print("fingerprint sha1[:12] =", corpus_fingerprint(CORPUS_PATH))
    raw = _read_json(CORPUS_PATH)
    print("raw records =", len(raw))
    df = pd.DataFrame([normalise(r) for r in raw])
    n0 = len(df)

    if LANG_SCOPE == "DE":                       # exclude undatable Rechtssätze
        n = int((~df.keep_type).sum())
        df = df[df.keep_type].copy()
        print("excluded %d RIS Rechtssätze (no single decision date -> undatable principle)" % n)

    flag  = DROP_COMMUNICATED if LANG_SCOPE == "EN" else DROP_CRIMINAL
    label = "DROP_COMMUNICATED"  if LANG_SCOPE == "EN" else "DROP_CRIMINAL"
    what  = "communicated-case templates" if LANG_SCOPE == "EN" else "criminal 'Entfremdung' (misappropriation) cases"
    n_conf = int(df.genre_confound.sum())
    if flag:
        df = df[~df.genre_confound].copy()
        print("%s=True -> removed %d %s" % (label, n_conf, what))
    else:
        print("%s=False -> %d %s flagged but kept" % (label, n_conf, what))

    nund = int(df.year.isna().sum())
    if nund:
        print("[!] %d records without a parseable year -> dropped" % nund)
    df = df[df.year.notna()].copy()
    df["year"] = df.year.astype(int)
    lo, hi = DATE_RANGE
    noor = int(((df.year < lo) | (df.year > hi)).sum())
    if noor:
        print("dropped %d records outside DATE_RANGE %s" % (noor, (lo, hi)))
    df = df[(df.year >= lo) & (df.year <= hi)].copy()

    df["bin"] = np.where(df.year < BIN_BOUNDARY, "early", "late")
    df["n_tokens"] = df.text.map(lambda t: len(tokens(t)))
    df["hits"] = df.text.map(cluster_count)
    df["mentions"] = df.hits > 0
    print("usable corpus: %d docs (of %d raw)" % (len(df), n0))
    return df.reset_index(drop=True)

df = load_corpus()
HAVE_DATA = not df.empty

LANG_SCOPE = EN -> ECHR (English judgments & decisions)
fingerprint sha1[:12] = 5413ea996016
raw records = 1116
DROP_COMMUNICATED=True -> removed 238 communicated-case templates
usable corpus: 878 docs (of 1116 raw)


In [ ]:
if HAVE_DATA:
    print("docs per YEAR:")
    for y, c in df.year.value_counts().sort_index().items():
        print("   %d : %d" % (y, c))
    bc = df.bin.value_counts()
    e, l = int(bc.get("early", 0)), int(bc.get("late", 0))
    print()
    print("docs per BIN (boundary=%d):" % BIN_BOUNDARY)
    print("   early %s : %d" % (EARLY_LABEL, e))
    print("   late  %s : %d" % (LATE_LABEL, l))
    print("   cluster-mentioning: early=%d  late=%d" % (
        int(df[(df.bin == 'early') & df.mentions].shape[0]),
        int(df[(df.bin == 'late') & df.mentions].shape[0])))
    for nm, n in [("early", e), ("late", l)]:
        if n < 20:
            print("   [!] WARNING: %s bin has only %d docs (<20) — the split may be too thin." % (nm, n))
else:
    print("[skipped: no data]")

docs per YEAR:
   2000 : 18
   2001 : 19
   2002 : 18
   2003 : 15
   2004 : 15
   2005 : 15
   2006 : 23
   2007 : 21
   2008 : 18
   2009 : 16
   2010 : 31
   2011 : 30
   2012 : 39
   2013 : 37
   2014 : 32
   2015 : 39
   2016 : 39
   2017 : 35
   2018 : 44
   2019 : 43
   2020 : 41
   2021 : 62
   2022 : 53
   2023 : 85
   2024 : 54
   2025 : 36

docs per BIN (boundary=2013):
   early 2000-2012 : 278
   late  2013-2025 : 600
   cluster-mentioning: early=104  late=226


## 4. Frequency + coverage over time (normalised, never raw)

Two normalised measures per year and per bin: **rate** = cluster occurrences per 10k tokens (controls for document length), and **coverage** = % of cases that mention the cluster at least once (controls for a few long cases dominating). N per bin is reported prominently; per-year bars carry their N because thin years are noisy. Both plots are saved to `figures/`.

In [ ]:
if HAVE_DATA:
    def agg_table(d, by):
        rows = []
        for key, sub in d.groupby(by):
            toks = int(sub.n_tokens.sum()); occ = int(sub.hits.sum()); nd = len(sub)
            rows.append({by: key, "n_docs": nd, "n_tokens": toks, "occ": occ,
                         "rate_per_10k": round(occ / toks * 10000, 3) if toks else 0.0,
                         "coverage_pct": round(int(sub.mentions.sum()) / nd * 100, 1) if nd else 0.0})
        return pd.DataFrame(rows).sort_values(by).reset_index(drop=True)

    by_year = agg_table(df, "year")
    by_bin  = agg_table(df, "bin")
    print("=== normalised frequency + coverage by BIN ===")
    print(by_bin.to_string(index=False))
    print()
    print("=== by YEAR (N shown; small-N years are noisy) ===")
    print(by_year.to_string(index=False))

    binN = dict(zip(by_bin["bin"], by_bin["n_docs"]))
    ne, nl = int(binN.get("early", 0)), int(binN.get("late", 0))
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
    ax[0].bar(by_year.year, by_year.rate_per_10k, color="#3b4a6b")
    ax[0].set_title("Cluster rate (occurrences / 10k tokens)"); ax[0].set_xlabel("year"); ax[0].set_ylabel("per 10k")
    ax[1].bar(by_year.year, by_year.coverage_pct, color="#6b5a3b")
    ax[1].set_title("Coverage (% of cases mentioning cluster)"); ax[1].set_xlabel("year"); ax[1].set_ylabel("%")
    for a in ax:
        a.axvline(BIN_BOUNDARY - 0.5, ls="--", c="crimson", lw=1)
    for x, n, h in zip(by_year.year, by_year.n_docs, by_year.rate_per_10k):
        ax[0].text(x, h, str(n), ha="center", va="bottom", fontsize=7)
    fig.suptitle("Diachronic signal — %s   (early %s N=%d | late %s N=%d)" % (
        CFG["name"], EARLY_LABEL, ne, LATE_LABEL, nl), fontsize=11)
    fig.tight_layout()
    outp = FIGURES / ("diachronic_%s.png" % LANG_SCOPE.lower())
    fig.savefig(outp, dpi=120, bbox_inches="tight")
    print("\nsaved figure ->", outp)
    plt.show()
else:
    print("[skipped: no data]")

=== normalised frequency + coverage by BIN ===
  bin  n_docs  n_tokens  occ  rate_per_10k  coverage_pct
early     278   2340650  242         1.034          37.4
 late     600   5889352  675         1.146          37.7

=== by YEAR (N shown; small-N years are noisy) ===
 year  n_docs  n_tokens  occ  rate_per_10k  coverage_pct
 2000      18    188756   20         1.060          50.0
 2001      19    169994   15         0.882          57.9
 2002      18    144407    9         0.623          22.2
 2003      15    143431   13         0.906          53.3
 2004      15    123585   24         1.942          33.3
 2005      15    120553   22         1.825          60.0
 2006      23    167313   12         0.717          34.8
 2007      21    190811   18         0.943          33.3
 2008      18    152460   11         0.722          22.2
 2009      16    107734    6         0.557          18.8
 2010      31    242949   14         0.576          25.8
 2011      30    268904   51         1.897    

## 5. KWIC concordances + collocation shift

KWIC shows the **immediate wording** around each anchor (one sample line per doc, per bin). Collocates show which content words sit within ±6 tokens of an anchor (stopwords and the anchors themselves removed) — a shift in the top collocates between bins is the visible framing move.

In [ ]:
if HAVE_DATA:
    def kwic_lines(d, n=8, width=45):
        out = []
        for _, row in d.iterrows():
            t = row.text
            for m in COMBINED.finditer(t):
                s = max(0, m.start() - width); e = min(len(t), m.end() + width)
                left  = re.sub(r"\s+", " ", t[s:m.start()])[-width:]
                mid   = t[m.start():m.end()]
                right = re.sub(r"\s+", " ", t[m.end():e])[:width]
                out.append("…%s [%s] %s…  (%s)" % (left, mid, right, row.id))
                break                                   # one line per doc, for variety
            if len(out) >= n:
                break
        return out

    def collocates(d, window=6, topn=15):
        cnt = Counter()
        for t in d.text:
            tk = tokens(t)
            for i, w in enumerate(tk):
                if not is_anchor_token(w):
                    continue
                for j in range(max(0, i - window), min(len(tk), i + window + 1)):
                    if j == i:
                        continue
                    x = tk[j]
                    if x in STOP or is_anchor_token(x) or x.isdigit() or len(x) <= 2:
                        continue
                    cnt[x] += 1
        return cnt.most_common(topn)

    for b, lab in [("early", EARLY_LABEL), ("late", LATE_LABEL)]:
        sub = df[(df.bin == b) & df.mentions]
        print("=" * 66)
        print("BIN %s (%s) — %d mentioning docs" % (b.upper(), lab, len(sub)))
        print("-- KWIC (sample, one line per doc) --")
        for ln in kwic_lines(sub):
            print("  ", ln)
        print("-- top collocates (±6 tokens, stopwords removed) --")
        print("  ", ", ".join("%s(%d)" % (w, c) for w, c in collocates(sub)))
        print()
else:
    print("[skipped: no data]")

BIN EARLY (2000-2012) — 104 mentioning docs
-- KWIC (sample, one line per doc) --
   …or that he was suffering from the so-called “ [parental alienation]  syndrome”.The expert considered that it was …  (001-105026)
   …ersistent pressure by the mother may lead to  [Parental Alienation]  Syndrome.53. On 26 February 2008 the first …  (001-106441)
   …amily had been affected by Gardner syndrome ( [Parental Alienation]  Syndrome)_ consisting in a strong emotional …  (001-110482)
   …decisions ordering access had contributed to  [alienating]  F. from him. 2. The applicant complains und…  (001-21924)
   …his mother's will_ this would put him into a  [loyalty conflict]  which he could not cope with and which would…  (001-58763)
   …n and that the children were suffering from “ [parental alienation] ” syndrome_ a condition recognised by the int…  (001-60163)
   …ven more by recent research on the so-called  [parental alienation]  syndrome (“PAS”)_ which has been described b…  (001-61195)
   …n

## 6. Keyness (log-likelihood G²)

Within the cluster's context, which terms are over-represented in the later bin versus the earlier? G² (Dunning's log-likelihood) is implemented by hand — no scipy — and the statistic is **signed**: positive = over-represented LATE, negative = EARLY. Thresholds: |G²| > 3.84 ≈ p<.05, |G²| > 10.83 ≈ p<.001.

`KEYNESS_SCOPE="window"` (default) scores only the ±`KEYNESS_WINDOW`-token neighbourhood of the anchors, so the result reflects the **framing around parental alienation**. `KEYNESS_SCOPE="document"` scores whole mentioning documents — broader, but **sensitive to corpus composition**: a late surge of climate/migration cases that merely contain a generic anchor (e.g. *undermine*), or the property-law *alienation* homonym, will dominate and swamp the framing signal. The full ranked table is written to `reports/`.

In [ ]:
KEYNESS_SCOPE  = "window"    # "window" = terms near the anchors (on-topic framing); "document" = whole mentioning docs (composition-sensitive)
KEYNESS_WINDOW = 15          # tokens each side of an anchor when scope == "window"

if HAVE_DATA:
    def bin_counts(d, which):
        cnt = Counter()
        sub = d[d.bin == which]
        if KEYNESS_SCOPE == "document":
            for t in sub.text:
                for w in tokens(t):
                    if w in STOP or w.isdigit() or len(w) <= 2:
                        continue
                    cnt[w] += 1
        else:
            W = KEYNESS_WINDOW
            for t in sub.text:
                tk = tokens(t)
                anchors = [i for i, w in enumerate(tk) if is_anchor_token(w)]
                for i in anchors:
                    for j in range(max(0, i - W), min(len(tk), i + W + 1)):
                        if j == i:
                            continue
                        w = tk[j]
                        if w in STOP or is_anchor_token(w) or w.isdigit() or len(w) <= 2:
                            continue
                        cnt[w] += 1
        return cnt

    ment = df[df.mentions]
    early_c, late_c = bin_counts(ment, "early"), bin_counts(ment, "late")
    c, d2 = sum(late_c.values()), sum(early_c.values())     # corpus totals
    rows = []
    for w in set(early_c) | set(late_c):
        a, b = late_c.get(w, 0), early_c.get(w, 0)          # a=late, b=early
        if a + b < 5:                                       # ignore very rare terms
            continue
        E1 = c * (a + b) / (c + d2); E2 = d2 * (a + b) / (c + d2)
        ll = 0.0
        if a > 0: ll += a * math.log(a / E1)
        if b > 0: ll += b * math.log(b / E2)
        g2 = 2 * ll
        late_rate = a / c if c else 0.0; early_rate = b / d2 if d2 else 0.0
        signed = g2 if late_rate >= early_rate else -g2
        rows.append({"term": w, "early": b, "late": a,
                     "G2": round(g2, 2), "signed_G2": round(signed, 2)})
    key = pd.DataFrame(rows).sort_values("signed_G2", ascending=False).reset_index(drop=True)

    scope_note = ("scope=window (±%d tokens around anchors)" % KEYNESS_WINDOW) if KEYNESS_SCOPE == "window" else "scope=document (whole mentioning docs)"
    print("keyness LATE (%s) vs EARLY (%s) — %s" % (LATE_LABEL, EARLY_LABEL, scope_note))
    print("|G2|>3.84 ~ p<.05 ; |G2|>10.83 ~ p<.001 ; +signed = over-represented LATE")
    print("late tokens=%d  early tokens=%d  scored vocab=%d" % (c, d2, len(key)))
    print()
    print("--- top 20 over-represented in LATE ---")
    print(key.head(20).to_string(index=False))
    print()
    print("--- top 20 over-represented in EARLY ---")
    print(key.tail(20).iloc[::-1].to_string(index=False))
    outp = REPORTS / ("keyness_%s_%s_%s_vs_%s.csv" % (LANG_SCOPE.lower(), KEYNESS_SCOPE, EARLY_LABEL, LATE_LABEL))
    key.to_csv(outp, index=False)
    print("\nsaved ranked keyness table ->", outp)
else:
    print("[skipped: no data]")

keyness LATE (2013-2025) vs EARLY (2000-2012) — scope=window (±15 tokens around anchors)
|G2|>3.84 ~ p<.05 ; |G2|>10.83 ~ p<.001 ; +signed = over-represented LATE
late tokens=8365  early tokens=3058  scored vocab=536

--- top 20 over-represented in LATE ---
         term  early  late    G2  signed_G2
        first      5    54 12.58      12.58
     domestic      3    37  9.65       9.65
      parent_      0    15  9.35       9.35
      contact     24   123  9.06       9.06
        moral      0    14  8.72       8.72
    behaviour      3    34  8.27       8.27
         need      0    13  8.10       8.10
  development      2    27  7.54       7.54
       unless      0    12  7.48       7.48
    including      0    12  7.48       7.48
        taken      0    12  7.48       7.48
       action      0    11  6.85       6.85
     measures      0    11  6.85       6.85
   concerning      1    19  6.53       6.53
reunification      0    10  6.23       6.23
      resides      0    10  6.23      

## 7. Appendix (optional, exploratory): contextual embedding drift

**Off by default.** Word2vec from scratch would be noise on a corpus this small, so the only variant here reuses `intfloat/multilingual-e5-base` to embed the sentences that contain a cluster anchor and compares the per-bin **centroid cosine**. It skips cleanly if sentence-transformers is absent or a bin has fewer than ~30 cluster-bearing sentences. Any output is a **hypothesis to verify by reading cases**, never a headline.

In [ ]:
RUN_EMBEDDING_APPENDIX = False    # opt-in; exploratory only. True -> embeds on CPU (~minutes).
SENT_CAP = 200                    # max cluster-bearing sentences embedded per bin

if not HAVE_DATA:
    print("[skipped: no data]")
elif not RUN_EMBEDDING_APPENDIX:
    print("appendix OFF (RUN_EMBEDDING_APPENDIX=False). The core analysis above is the result.")
    print("Set RUN_EMBEDDING_APPENDIX=True to compute exploratory per-bin centroid drift.")
else:
    try:
        from sentence_transformers import SentenceTransformer
    except Exception as ex:
        SentenceTransformer = None
        print("sentence-transformers unavailable -> skipping appendix:", ex)
    if SentenceTransformer is not None:
        _SENT = re.compile(r"(?<=[.!?])\s+")
        def cluster_sentences(d):
            ss = []
            for t in d.text:
                for s in _SENT.split(t):
                    s = re.sub(r"\s+", " ", s).strip()
                    if 20 <= len(s) <= 400 and COMBINED.search(s):
                        ss.append(s)
            return ss
        se = cluster_sentences(df[df.bin == "early"])[:SENT_CAP]
        sl = cluster_sentences(df[df.bin == "late"])[:SENT_CAP]
        print("cluster-bearing sentences: early=%d late=%d (cap=%d)" % (len(se), len(sl), SENT_CAP))
        if min(len(se), len(sl)) < 30:
            print("[skip] fewer than 30 cluster sentences in a bin -> centroid drift not meaningful.")
        else:
            model = SentenceTransformer("intfloat/multilingual-e5-base")
            ee = model.encode(["passage: " + s for s in se], normalize_embeddings=True, batch_size=16, show_progress_bar=False)
            el = model.encode(["passage: " + s for s in sl], normalize_embeddings=True, batch_size=16, show_progress_bar=False)
            ce = ee.mean(0); cl = el.mean(0)
            ce = ce / (np.linalg.norm(ce) + 1e-9); cl = cl / (np.linalg.norm(cl) + 1e-9)
            print("centroid cosine(early, late) = %.4f  (1.0 = no drift; lower = more contextual drift)" % float(np.dot(ce, cl)))
            print("[exploratory] a HYPOTHESIS of framing drift to verify by reading the cases the")
            print("              keyness / KWIC shortlists flag — not evidence on its own.")

appendix OFF (RUN_EMBEDDING_APPENDIX=False). The core analysis above is the result.
Set RUN_EMBEDDING_APPENDIX=True to compute exploratory per-bin centroid drift.


## 8. From signals to doctrine — how these outputs feed validation

Pipeline: **signal → shortlist → read**. None of the steps below is evidence on its own; each narrows hundreds of cases to a readable shortlist.

| Output | Signals what | Feeds | Research question |
|---|---|---|---|
| Frequency (occ/10k) + coverage (%) per bin | whether the alienation vocabulary became **more prevalent** | which bin/years to inspect; N per bin | **RQ1** — did the alienation framing become more *salient* over time? |
| KWIC concordances | the **immediate phrasing** around each anchor, per bin | hand-reading of representative lines | **RQ2** — how is alienation *characterised*, and did the wording move? |
| Collocation shift (±6) | which **concepts co-occur** with the anchors, per bin | candidate framing partners (e.g. *contact*, *manipulation*, *best interests*) | **RQ2** |
| Keyness G² (signed) | terms **statistically over-represented** in the later bin's alienation discourse | a ranked shortlist of candidate new framings | **RQ2 / RQ3** |
| Embedding drift (appendix) | exploratory hypothesis of **contextual** drift | whether to look harder; never a headline | **RQ3** (exploratory) |

**Validation step.** Take the top signed-G² terms and the KWIC/collocation shortlist, pull the specific later-bin cases (by `id`) where those terms occur, and read them to decide whether the **legal interpretation** of parental alienation actually shifted — or whether the signal reflects drafting style, a single influential judgment, or a change in corpus composition. The notebook produces the shortlist; the doctrinal claim is made by the reader.